# Tien schaalkeuzes vastzetten na epoch 50 — vijf epochs voortzetten

**Onderzoeksvraag:** helpt het de verdere gewichtstraining als de geleerde keuze uit tien schaalfactoren per MX-groep vanaf epoch 51 vast blijft?

We laden het bestaande checkpoint na **epoch 50** van de voltooide training met tien opties en trainen alleen de variant met vaste offsets verder, tot **epoch 55**. De vergelijking gebruikt de al gemeten epochs 51–55 uit diezelfde oorspronkelijke training. De eerste 50 epochs worden geërfd en tellen **niet** als nieuwe training; de historische doorlerende variant wordt niet opnieuw getraind.

De tien mogelijke factoren zijn **×1, ×2, ×4, ×8, ×16, ×32, ×64, ×128, ×256 en ×512**. De route blijft **E4M3 → E2M3 → E2M1**, met groepen van 32 gewichten, **A32**-activaties en **FP32**-mastergewichten. E2M3 behoudt de hoogste twee exponentbits van de E4M3-bron; E2M1 ontstaat via mantissatruncatie en deelt de lagere schaal met E2M3.

Deze notebook leest opgeslagen meetresultaten en controleverslagen. Zij laadt geen model/checkpoint en start geen training. Alleen `SAVE_FIGURES = True` schrijft figuren in een nieuwe analysemap. De bronbestanden blijven behouden.


In [ ]:
from pathlib import Path
RUN_DIR = Path('C:\\Users\\renzo\\master-thesis-mx\\mx_runs\\mx_triple_freeze_scale10_offsets_e50_pilot5\\20261109_183755_676244_freeze_check')
SAVE_FIGURES = False  # True: bewaar figuren in een nieuwe analysemap.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit deze run; vermijd een module uit een eerder geopende notebook.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
if not helper_path.is_file():
    raise FileNotFoundError('Stel RUN_DIR in op de resultatenmap van de tienkeuze-freezeproef: ' + str(helper_path.parent))
helper_name = '_mx_freeze_scale10_offsets_e50_pilot5_analysis'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(ns.diagnostics_table(run))


## Wat wordt vastgezet?

Voor iedere groep blijft de lagere shared exponent:

$$E_{\mathrm{laag},b}(t) = E_{\mathrm{E4M3},b}(t) + k_b.$$

De offset $k_b\in\{0,1,\ldots,9\}$ blijft gelijk aan de harde keuze na epoch 50. De schaalverhouding is $2^{k_b}$. **De E4M3-broncode en bronschaal worden nog steeds uit de actuele gewichten bepaald.** De uiteindelijke shared exponent kan dus veranderen terwijl de offset vast blijft. Dit is geen proef met volledig onafhankelijke schalen per exponentbreedte.

De FP32-schaalproxy's krijgen vanaf epoch 51 een learning rate van **nul**. De bestaande gradientberekening en Adam-momentupdates blijven actief. Na elke optimizerstap wordt gecontroleerd dat de proxywaarden én harde keuzes exact gelijk blijven. Deze diagnostische uitvoering claimt geen trainingsbesparing door schaalgradients te verwijderen.

Het eerste overzicht toont de geërfde epochs 1–50 en de voortzetting. De daaropvolgende figuur zoomt in op het gemeenschappelijke checkpoint na epoch 50 en de twee trajecten daarna.


In [ ]:
figures['accuracy_full_context'] = ns.plot_learning_context(run)
display(figures['accuracy_full_context'])
plt.close(figures['accuracy_full_context'])


In [ ]:
figures['accuracy_curves'] = ns.plot_accuracy_curves(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Primair vergelijkingspunt: epoch 55

We vergelijken alle drie de precisies op **dezelfde epoch 55**. Het verschil hieronder is **vaste offset minus verder geleerde offset**, uitgedrukt in procentpunt. Deze groep wordt pas getoond als epoch 55 daadwerkelijk is gemeten.


In [ ]:
display(ns.checkpoint_table(run, 'endpoint').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
delta = ns.delta_table(run)
if not delta.empty:
    display(delta.loc[delta.Epoch.eq(55)].drop(columns=['weight_bits']).round(6))
figures['accuracy_epoch55'] = ns.plot_checkpoint_accuracy(run, 'endpoint')
display(figures['accuracy_epoch55'])
plt.close(figures['accuracy_epoch55'])


## Aanvullend: beste checkpoint gekozen door E4M3

Per variant kiezen we het **eerste maximum van de E4M3-testaccuracy binnen epochs 51–55**. Vervolgens tonen we E4M3, E2M3 en E2M1 op precies diezelfde epoch. De varianten mogen hierbij elk een andere geselecteerde epoch hebben. Het globale beste checkpoint uit de oorspronkelijke 100-epochtraining wordt voor deze vergelijking niet gebruikt.

Bij een onderbroken voortzetting wordt alleen over het gemeenschappelijk gemeten deel van 51–55 geselecteerd en blijft de uitkomst voorlopig. Zoals in het bestaande protocol gebeurt de selectie op de officiële testset; de geselecteerde score is dus geen onafhankelijke validatiescore.


In [ ]:
display(ns.checkpoint_table(run, 'best').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_best_E4_within_window'] = ns.plot_checkpoint_accuracy(run, 'best')
display(figures['accuracy_best_E4_within_window'])
plt.close(figures['accuracy_best_E4_within_window'])


## Accuracyverschillen gedurende de voortzetting

Deze figuur gebruikt voor elke epoch en precisie de bijbehorende historische meting. De vijf epochs zijn opeenvolgende toestanden van één training, geen vijf onafhankelijke herhalingen. Positieve verschillen betekenen een hogere accuracy met vaste offsets; negatieve verschillen betekenen een lagere accuracy.


In [ ]:
if not delta.empty:
    display(delta.drop(columns=['weight_bits']).round(6))
figures['accuracy_delta'] = ns.plot_accuracy_delta(run)
display(figures['accuracy_delta'])
plt.close(figures['accuracy_delta'])


## Verdeling van de gekozen schaalfactoren per checkpoint

De twee staafdiagrammen tonen **het aantal groepen én het percentage per factor** voor het laatste checkpoint en het gezamenlijke beste E4M3-checkpoint binnen de nieuwe fase. **Alle tien factoren blijven zichtbaar, ook als nul groepen die factor gebruiken.** Percentages hebben steeds alle 4.080 groepen als noemer; ongebruikte factoren worden niet samengevoegd of weggelaten.

Bij een voltooide proef is het laatste checkpoint epoch 55. De E4M3-selectie kiest één checkpoint voor alle drie de precisies. Als beide rollen dezelfde epoch aanwijzen, staat dat bij de figuur. Omdat de offsets vanaf epoch 51 vast blijven, moeten de beide verdelingen in deze proef overeenkomen met het startcheckpoint na epoch 50.


In [ ]:
display(ns.factor_distribution_table(run, 'last').round({'Aandeel [%]': 4}))
display(ns.factor_distribution_table(run, 'best').round({'Aandeel [%]': 4}))
figures['scale_factor_counts_and_percentages'] = ns.plot_factor_distribution(run)
display(figures['scale_factor_counts_and_percentages'])
plt.close(figures['scale_factor_counts_and_percentages'])


## Controle: blijven alle tien mogelijke offsetkeuzes werkelijk vast?

We controleren zowel de opgeslagen FP32-proxywaarden als de harde keuzes. De nieuwe voortzetting moet **nul gewijzigde proxy's en nul offsetwissels** tonen. De verdeling over alle tien factoren blijft gelijk aan het checkpoint na epoch 50. Een constant totaalaandeel alleen volstaat niet: individuele groepen zouden in tegengestelde richting kunnen wisselen.

De teller ‘offsetwissels tijdens epoch’ telt gebeurtenissen. Een groep kan in de historische doorlerende variant meerdere keren binnen één epoch wisselen. De gekleurde verdelingen tonen de keuzes op de opgeslagen epochgrenzen.


In [ ]:
display(ns.freeze_table(run).round(6))
figures['scale_choices_and_switches'] = ns.plot_scale_choices(run)
display(figures['scale_choices_and_switches'])
plt.close(figures['scale_choices_and_switches'])


## Trainingsobjectief

E4M3 gebruikt harde cross-entropy. E2M3 en E2M1 gebruiken ieder zachte cross-entropy met de vorige precisie als losgekoppelde leraar. Vergelijk de twee varianten per precisie; de drie trainingsobjectieven zijn niet onderling gelijk aan harde test-cross-entropy.


In [ ]:
figures['training_objectives'] = ns.plot_objectives(run)
display(figures['training_objectives'])
plt.close(figures['training_objectives'])


## Gemeten duur en interpretatie

De epochtijden omvatten training, evaluatie, controles en checkpointschrijven. De nieuwe uitvoering start vanaf een checkpoint en kan andere opstart- of uitvoeringskosten hebben. De waarden hieronder zijn daarom beschrijvende metingen van deze runs. Dit is een FP32-emulatie van de MX-gewichten; ze meet geen native MX-versnelling of energieverbruik.


In [ ]:
display(ns.timing_table(run).round(3))
figures['epoch_timing'] = ns.plot_timing(run)
display(figures['epoch_timing'])
plt.close(figures['epoch_timing'])


## Kosten van tien keuzes en van deze bevriezingsmethode

Dit zijn **analytische extra payload- en tensoromvangen**, geen gemeten volledige checkpointgroottes, VRAM-pieken, energie of native MX-snelheid.

- Voor tien mogelijkheden zijn **4 keuzebits per groep** nodig. De opgeslagen nibble bevat offset 0–9; waarden 10–15 zijn ongeldig. Voor de 4.080 groepen is dat **2.040 bytes** keuzemetadata in de flexibele E4M3-bron, tegenover 510 bytes bij twee keuzes. De zes lagen hebben ieder een even aantal groepen, zodat hiervoor geen extra halve byte nodig is.
- De analytische FP32-proxy, gradient en twee Adam-momenttensors kosten samen **4.080 × 4 × 4 = 65.280 bytes (63,75 KiB)**. Dit telt alleen deze vier tensors; overige modeldata, Adam-staptellers, tijdelijke buffers en frameworkoverhead vallen erbuiten.
- **Freeze met een schaal-LR van nul verwijdert die tensors of berekeningen niet.** De diagnostische freeze heeft daardoor dezelfde analytische schaaltensorkosten als doorleren met tien keuzes. De gemeten epochtijden staan hierboven.
- Voor uitvoering van één al afgeleide lagere precisie kan de gekozen offset in de eigen E8M0-schaal worden verwerkt. De keuzemetadata hier betreft de flexibele bron waaruit meerdere modi worden afgeleid.


In [ ]:
display(ns.storage_cost_table(run))
figures['scale_choice_storage_cost'] = ns.plot_storage_cost(run)
display(figures['scale_choice_storage_cost'])
plt.close(figures['scale_choice_storage_cost'])


## Hoe beoordelen we de proef?

- Begin met de drie accuracies op **epoch 55** en hun verschillen met de bestaande dynamische training.
- Bekijk vervolgens of een verschil gedurende meerdere gemeten epochs in dezelfde richting gaat en of de controle daadwerkelijk nul offsetwissels bevestigt.
- Gebruik de E4M3-selectie als aanvullend overzicht: alle drie de modi blijven gekoppeld aan één checkpoint per variant.
- De bestaande CUDA-instellingen garanderen geen bitexacte herhaling. Eén voortzetting geeft geen schatting van de spreiding tussen runs; een klein verschil verdient voorzichtigheid.
- Een gunstige uitkomst ondersteunt verder onderzoek naar het vastzetten van de offsetkeuzes. Deze proef bepaalt op zichzelf geen optimaal moment om daarmee te beginnen.

De referentiebestanden en hun herkomst zijn meegeleverd onder `parent_reference`. De analyse verifieert hun hashes en vergelijkt de opgeslagen train-/testvolgorde met die van de oorspronkelijke epochs 51–55.


In [ ]:
if SAVE_FIGURES:
    destination = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', destination)
else:
    print('Alleen gelezen en weergegeven; bestaande resultaten zijn ongewijzigd.')
